In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import batman
import emcee
import corner

# load K2 data
data = np.loadtxt(
    "../data/apjladf06ft3_mrt.txt",
    skiprows=15
)

t = data[:, 0] + 2454833  # offset to full BJD
flux = data[:, 2]

# published mid-transit time from Venner et al. (2026)
t0_published = 2458055.0969

# only use the region around the transit
mask = np.abs(t - t0_published) < 0.8

t_transit = t[mask]
flux_transit = flux[mask]

In [ ]:
# gravitational constant
G = 6.67430e-11

# scatter of the K2 light curve outside the transit
sigma_flux = 32e-6

# stellar-density prior
logrho_mean = 0.462
logrho_sigma = 0.041

# limb-darkening priors
u1_mean = 0.60
u1_sigma = 0.07

u2_mean = 0.12
u2_sigma = 0.07

# keep the same maximum eccentricity as in notebook 04
e_max = 0.8

# physically motivated low-e prior:
# exponential distribution with mean eccentricity = 0.06
e_mean = 0.06

In [ ]:
def a_over_rstar(period, rho_star):
    """
    Calculate a/R_star from the period and stellar density.
    """

    # days -> seconds
    period_seconds = period * 24 * 60 * 60

    # g/cm^3 -> kg/m^3
    rho_si = rho_star * 1000

    return (
        G * rho_si * period_seconds**2
        / (3 * np.pi)
    )**(1/3)

In [ ]:
def eccentricity_from_hk(h, k):
    """
    h = sqrt(e) cos(omega)
    k = sqrt(e) sin(omega)
    """

    e = h**2 + k**2

    omega = np.degrees(
        np.arctan2(k, h)
    ) % 360

    return e, omega

In [ ]:
batman_params = batman.TransitParams()

batman_params.t0 = t0_published
batman_params.per = 355.0
batman_params.rp = 0.01368
batman_params.a = 270.0
batman_params.inc = 90.0

batman_params.ecc = 0.0
batman_params.w = 90.0

batman_params.u = [0.60, 0.12]
batman_params.limb_dark = "quadratic"

batman_model = batman.TransitModel(
    batman_params,
    t_transit,
    supersample_factor=10,
    exp_time=29.4 / (24 * 60)
)

In [ ]:
def transit_model(theta):
    """
    Eccentric transit model.

    theta =
    (period, b, rp, t0, rho_star, u1, u2, h, k)
    """

    period, b, rp, t0, rho_star, u1, u2, h, k = theta

    e, omega = eccentricity_from_hk(h, k)

    a_Rstar = a_over_rstar(period, rho_star)

    omega_rad = np.radians(omega)

    # impact parameter relation for an eccentric orbit
    cos_i = (
        b / a_Rstar
        * (1 + e * np.sin(omega_rad))
        / (1 - e**2)
    )

    if cos_i < 0 or cos_i > 1:
        return None

    inclination = np.degrees(
        np.arccos(cos_i)
    )

    batman_params.t0 = t0
    batman_params.per = period
    batman_params.rp = rp
    batman_params.a = a_Rstar
    batman_params.inc = inclination

    batman_params.ecc = e
    batman_params.w = omega

    batman_params.u = [u1, u2]

    return batman_model.light_curve(
        batman_params
    )

In [ ]:
def log_prior(theta):
    """
    Priors for the low-eccentricity model.

    The only important change compared with notebook 04
    is the eccentricity prior.
    """

    period, b, rp, t0, rho_star, u1, u2, h, k = theta

    e, omega = eccentricity_from_hk(h, k)


    # --------------------------------------------------
    # basic parameter limits
    # --------------------------------------------------

    if not 64 < period < 2000:
        return -np.inf

    if not 0 <= b < 1 + rp:
        return -np.inf

    if not 0.005 < rp < 0.03:
        return -np.inf

    if not t0_published - 0.1 < t0 < t0_published + 0.1:
        return -np.inf

    if rho_star <= 0:
        return -np.inf

    if not 0 <= e < e_max:
        return -np.inf


    # --------------------------------------------------
    # physical limb-darkening conditions
    # --------------------------------------------------

    if not (
        u1 > 0
        and u1 + u2 < 1
        and u1 + 2*u2 > 0
    ):
        return -np.inf


    # --------------------------------------------------
    # orbital geometry
    # --------------------------------------------------

    a_Rstar = a_over_rstar(
        period,
        rho_star
    )

    # periastron cannot be inside the star
    if a_Rstar * (1 - e) <= 1 + rp:
        return -np.inf

    omega_rad = np.radians(omega)

    cos_i = (
        b / a_Rstar
        * (1 + e * np.sin(omega_rad))
        / (1 - e**2)
    )

    if cos_i < 0 or cos_i > 1:
        return -np.inf


    lp = 0.0


    # --------------------------------------------------
    # same period prior as notebooks 03 and 04
    # --------------------------------------------------

    lp += -np.log(period)

    # approximate Kipping single-transit prior
    lp += -(5/3) * np.log(period)


    # --------------------------------------------------
    # Rp/Rstar prior
    # --------------------------------------------------

    lp += -np.log(rp)


    # --------------------------------------------------
    # stellar-density prior
    # --------------------------------------------------

    lp += -np.log(rho_star)

    log_rho = np.log10(rho_star)

    lp += -0.5 * (
        (log_rho - logrho_mean)
        / logrho_sigma
    )**2


    # --------------------------------------------------
    # limb-darkening priors
    # --------------------------------------------------

    lp += -0.5 * (
        (u1 - u1_mean)
        / u1_sigma
    )**2

    lp += -0.5 * (
        (u2 - u2_mean)
        / u2_sigma
    )**2


    # --------------------------------------------------
    # NEW: low-eccentricity prior
    # --------------------------------------------------

    # exponential eccentricity distribution
    # with mean e = 0.06
    lp += -e / e_mean


    return lp

In [ ]:
def log_likelihood(theta):

    model_flux = transit_model(theta)

    if model_flux is None:
        return -np.inf

    if not np.all(
        np.isfinite(model_flux)
    ):
        return -np.inf

    residuals = (
        flux_transit - model_flux
    )

    return -0.5 * np.sum(
        residuals**2 / sigma_flux**2
        + np.log(
            2 * np.pi
            * sigma_flux**2
        )
    )

In [ ]:
def log_probability(theta):

    lp = log_prior(theta)

    if not np.isfinite(lp):
        return -np.inf

    return (
        lp
        + log_likelihood(theta)
    )

In [ ]:
e_test = 0.05
omega_test = 90.0

h_test = (
    np.sqrt(e_test)
    * np.cos(
        np.radians(omega_test)
    )
)

k_test = (
    np.sqrt(e_test)
    * np.sin(
        np.radians(omega_test)
    )
)

theta_test = (
    355.0,
    0.3,
    0.01368,
    t0_published,
    2.90,
    0.60,
    0.12,
    h_test,
    k_test
)

print(
    "e, omega:",
    eccentricity_from_hk(
        h_test,
        k_test
    )
)

print(
    "log prior:",
    log_prior(theta_test)
)

print(
    "log likelihood:",
    log_likelihood(theta_test)
)

print(
    "log posterior:",
    log_probability(theta_test)
)

In [ ]:
ndim = 9
nwalkers = 60

labels = [
    "Period [d]",
    "b",
    "Rp/Rstar",
    "t0 [BJD]",
    "rho_star",
    "u1",
    "u2",
    "h",
    "k"
]

initial = np.array([
    355.0,
    0.3,
    0.01368,
    t0_published,
    2.90,
    0.60,
    0.12,
    h_test,
    k_test
])

spread = np.array([
    30.0,
    0.15,
    0.0005,
    0.005,
    0.10,
    0.03,
    0.03,
    0.08,
    0.08
])

rng = np.random.default_rng(42)

In [ ]:
pos = []

while len(pos) < nwalkers:

    candidate = (
        initial
        + spread
        * rng.normal(size=ndim)
    )

    # draw a starting eccentricity from
    # approximately the low-e prior
    e_start = rng.exponential(
        e_mean
    )

    # make sure it lies inside our allowed range
    if e_start >= e_max:
        continue

    omega_start = rng.uniform(
        0,
        2 * np.pi
    )

    candidate[7] = (
        np.sqrt(e_start)
        * np.cos(omega_start)
    )

    candidate[8] = (
        np.sqrt(e_start)
        * np.sin(omega_start)
    )

    if np.isfinite(
        log_probability(candidate)
    ):
        pos.append(candidate)

pos = np.array(pos)

print(pos.shape)

In [ ]:
sampler = emcee.EnsembleSampler(
    nwalkers,
    ndim,
    log_probability
)

sampler.run_mcmc(
    pos,
    10000,
    progress=True
)

In [ ]:
print(
    "Mean acceptance fraction:",
    np.mean(
        sampler.acceptance_fraction
    )
)

tau = sampler.get_autocorr_time(
    tol=0
)

print()

for label, value in zip(
    labels,
    tau
):
    print(
        f"{label}: "
        f"tau = {value:.1f}, "
        f"N/tau = "
        f"{sampler.iteration/value:.1f}"
    )

In [ ]:
chain = sampler.get_chain()

fig, axes = plt.subplots(
    ndim,
    figsize=(10, 16),
    sharex=True
)

for i in range(ndim):

    axes[i].plot(
        chain[:, :, i],
        alpha=0.20
    )

    axes[i].set_ylabel(
        labels[i]
    )

axes[-1].set_xlabel(
    "MCMC step"
)

plt.tight_layout()
plt.show()

In [ ]:
chain = sampler.get_chain()

N = sampler.iteration

start = N // 4
middle = (start + N) // 2

samples_earlier = (
    chain[start:middle]
    .reshape(-1, ndim)
)

samples_later = (
    chain[middle:N]
    .reshape(-1, ndim)
)

for i, label in enumerate(labels):

    q1 = np.percentile(
        samples_earlier[:, i],
        [16, 50, 84]
    )

    q2 = np.percentile(
        samples_later[:, i],
        [16, 50, 84]
    )

    print(label)
    print("  earlier:", q1)
    print("  later:  ", q2)

In [ ]:
burnin = sampler.iteration // 2
thin = 100

lowe_samples = sampler.get_chain(
    discard=burnin,
    thin=thin,
    flat=True
)

print(
    "Total chain length:",
    sampler.iteration
)

print(
    "Posterior samples:",
    len(lowe_samples)
)

In [ ]:
np.save(
    "../data/posterior_eccentric_lowe.npy",
    lowe_samples
)

In [ ]:
circular_samples = np.load(
    "../data/posterior_circular.npy"
)

uniform_samples = np.load(
    "../data/posterior_eccentric_uniform.npy"
)

lowe_samples = np.load(
    "../data/posterior_eccentric_lowe.npy"
)

print(
    circular_samples.shape,
    uniform_samples.shape,
    lowe_samples.shape
)

In [ ]:
period_circular = (
    circular_samples[:, 0]
)

period_uniform = (
    uniform_samples[:, 0]
)

period_lowe = (
    lowe_samples[:, 0]
)

In [ ]:
bins = np.linspace(
    50,
    1000,
    80
)

plt.figure(figsize=(9, 5))

plt.hist(
    period_circular,
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2,
    label="Circular: e = 0"
)

plt.hist(
    period_lowe,
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2,
    label="Low-e prior"
)

plt.hist(
    period_uniform,
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2,
    label="Broad e prior"
)

plt.xlabel(
    "Orbital period [days]"
)

plt.ylabel(
    "Posterior density"
)

plt.title(
    "Sensitivity of orbital period to eccentricity prior"
)

plt.xlim(50, 1000)

plt.legend()
plt.show()

In [ ]:
def summary(samples):

    q16, q50, q84 = np.percentile(
        samples,
        [16, 50, 84]
    )

    return (
        q50,
        q50 - q16,
        q84 - q50
    )


circ_summary = summary(
    period_circular
)

lowe_summary = summary(
    period_lowe
)

uniform_summary = summary(
    period_uniform
)

print(
    "Circular:",
    circ_summary
)

print(
    "Low-e:",
    lowe_summary
)

print(
    "Broad-e:",
    uniform_summary
)

In [ ]:
print(
    f"Circular: "
    f"{circ_summary[0]:.1f} "
    f"-{circ_summary[1]:.1f} "
    f"+{circ_summary[2]:.1f} d"
)

print(
    f"Low-e: "
    f"{lowe_summary[0]:.1f} "
    f"-{lowe_summary[1]:.1f} "
    f"+{lowe_summary[2]:.1f} d"
)

print(
    f"Broad-e: "
    f"{uniform_summary[0]:.1f} "
    f"-{uniform_summary[1]:.1f} "
    f"+{uniform_summary[2]:.1f} d"
)

In [ ]:
e_circular = np.zeros(
    len(circular_samples)
)
e_uniform = (
    uniform_samples[:, 7]**2
    + uniform_samples[:, 8]**2
)

e_lowe = (
    lowe_samples[:, 7]**2
    + lowe_samples[:, 8]**2
)

In [ ]:
bins_e = np.linspace(
    0,
    0.8,
    50
)

plt.figure(figsize=(8, 5))

plt.hist(
    e_lowe,
    bins=bins_e,
    density=True,
    histtype="step",
    linewidth=2,
    label="Low-e prior"
)

plt.hist(
    e_uniform,
    bins=bins_e,
    density=True,
    histtype="step",
    linewidth=2,
    label="Broad e prior"
)

plt.xlabel("Eccentricity")
plt.ylabel("Posterior density")

plt.title(
    "Eccentricity posterior under different priors"
)

plt.legend()
plt.show()

In [ ]:
M_sun = 1.98847e30
AU = 1.495978707e11

Mstar_mean = 0.726
Mstar_sigma = 0.017


def semi_major_axis(
    period_days,
    rng
):
    """
    Calculate semi-major axis in AU.

    For every posterior period sample we also
    draw a stellar mass from its uncertainty.
    """

    Mstar = rng.normal(
        Mstar_mean,
        Mstar_sigma,
        size=len(period_days)
    )

    P_seconds = (
        period_days
        * 24 * 60 * 60
    )

    a_m = (
        G
        * (Mstar * M_sun)
        * P_seconds**2
        / (4 * np.pi**2)
    )**(1/3)

    return a_m / AU

In [ ]:
rng = np.random.default_rng(123)

a_circular = semi_major_axis(
    period_circular,
    rng
)

a_lowe = semi_major_axis(
    period_lowe,
    rng
)

a_uniform = semi_major_axis(
    period_uniform,
    rng
)

In [ ]:
bins_a = np.linspace(
    0.2,
    1.8,
    70
)

plt.figure(figsize=(9, 5))

plt.hist(
    a_circular,
    bins=bins_a,
    density=True,
    histtype="step",
    linewidth=2,
    label="Circular"
)

plt.hist(
    a_lowe,
    bins=bins_a,
    density=True,
    histtype="step",
    linewidth=2,
    label="Low-e prior"
)

plt.hist(
    a_uniform,
    bins=bins_a,
    density=True,
    histtype="step",
    linewidth=2,
    label="Broad e prior"
)

plt.xlabel(
    "Semi-major axis [AU]"
)

plt.ylabel(
    "Posterior density"
)

plt.title(
    "Effect of eccentricity prior on orbital separation"
)

plt.legend()
plt.show()

In [ ]:
Lstar_mean = 0.232

# use approximately symmetric uncertainty
Lstar_sigma = 0.022

In [ ]:
def average_flux(
    a,
    e,
    rng
):
    """
    Orbit-averaged stellar flux relative to Earth.
    """

    Lstar = rng.normal(
        Lstar_mean,
        Lstar_sigma,
        size=len(a)
    )

    return (
        Lstar
        / (
            a**2
            * np.sqrt(1 - e**2)
        )
    )

In [ ]:
rng = np.random.default_rng(456)

flux_circular = average_flux(
    a_circular,
    e_circular,
    rng
)

flux_lowe = average_flux(
    a_lowe,
    e_lowe,
    rng
)

flux_uniform = average_flux(
    a_uniform,
    e_uniform,
    rng
)

In [ ]:
bins_flux = np.linspace(
    0,
    2,
    80
)

plt.figure(figsize=(9, 5))

plt.hist(
    flux_circular,
    bins=bins_flux,
    density=True,
    histtype="step",
    linewidth=2,
    label="Circular"
)

plt.hist(
    flux_lowe,
    bins=bins_flux,
    density=True,
    histtype="step",
    linewidth=2,
    label="Low-e prior"
)

plt.hist(
    flux_uniform,
    bins=bins_flux,
    density=True,
    histtype="step",
    linewidth=2,
    label="Broad e prior"
)

plt.xlabel(
    r"Orbit-averaged incident flux "
    r"[$S_\oplus$]"
)

plt.ylabel(
    "Posterior density"
)

plt.title(
    "Effect of eccentricity prior on irradiation" 
)

plt.legend()
plt.show()

In [ ]:
for name, samples in [
    ("Circular", flux_circular),
    ("Low-e", flux_lowe),
    ("Broad-e", flux_uniform)
]:

    q16, q50, q84 = np.percentile(
        samples,
        [16, 50, 84]
    )

    print(
        f"{name}: "
        f"{q50:.3f} "
        f"-{q50-q16:.3f} "
        f"+{q84-q50:.3f} S_earth"
    )